# calm forest · 복귀율(리텐션) EDA

**목적**: D1 복귀율(2026-08-13 기준 9.1%)의 원인 탐색과, 밤손님·날씨 이벤트 배포 전후 비교의 베이스라인 확보.

- 데이터: Supabase `session_logs` / `econ_logs` / `game_logs` (접속은 `calm_ml.db`가 루트 `.env`를 읽어 처리)
- 실험 기록: WandB `calm-forest` 프로젝트 (`calm_ml.tracking.start_run`)
- 유저 단위: `client_id`(기기) — 게스트 포함 리텐션을 재기 위함

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from calm_ml import db

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "AppleGothic"   # macOS 한글 폰트
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

## 1. 일별 활성 기기(DAU)

In [ ]:
dau = db.load_daily_active(days=90)
ax = dau.plot(x="day", y="dau", marker="o", figsize=(10, 3), legend=False)
ax.set_title("일별 활성 기기 수"); ax.set_xlabel(""); ax.set_ylabel("기기")
dau.tail()

## 2. D1 복귀 — 첫 방문일 코호트

In [ ]:
co = db.load_d1_cohorts()
co["d1_rate"] = co.returned_d1 / co.new_users
print(f"전체 D1: {co.returned_d1.sum()}/{co.new_users.sum()} = {co.returned_d1.sum()/co.new_users.sum():.1%}")
ax = co.plot(x="cohort", y="d1_rate", kind="bar", figsize=(10, 3), legend=False)
ax.set_title("코호트별 D1 복귀율"); ax.set_xlabel("첫 방문일"); ax.set_ylabel("D1")
co

## 3. 세션 테이블 훑어보기
게스트/로그인, A/B variant 세그먼트 분포 확인.

In [ ]:
ses = db.load_sessions(days=90)
display(ses.head())
ses.groupby(["is_guest", "variant"]).agg(
    sessions=("session_id", "nunique"), devices=("client_id", "nunique")
)

## 4. 경제 원장 — 코인 유입/유출 출처

In [ ]:
econ = db.load_econ(days=90)
flow = econ.groupby("source").amount.agg(["sum", "count"]).sort_values("sum", ascending=False)
flow

## 5. WandB 에 베이스라인 기록
밤손님·날씨 이벤트 **배포 전** 지표를 런으로 남겨, 배포 후와 비교한다.
(처음이면 터미널에서 `uv run wandb login` 1회)

In [ ]:
from calm_ml.tracking import start_run

with start_run("retention-baseline", tags=["eda", "pre-night-visit"]) as run:
    run.log({
        "d1_rate": co.returned_d1.sum() / co.new_users.sum(),
        "new_users": int(co.new_users.sum()),
        "dau_last": int(dau.dau.iloc[-1]),
    })
print("기록 완료")